In [ ]:
#pydantic model for evaluation
from pydantic import BaseModel


In [17]:
#pydantic model for evaluation
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from pypdf import PdfReader
import gradio as gr

In [4]:
load_dotenv(override=True)
openai = OpenAI()

In [5]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [6]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [7]:
name = "Javier Lopez"

In [8]:
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [9]:
#se define el evaluador

evaluator_system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

#Como lo adaptare en un futuro

# system_prompt = f"Actuas como {name} cuyo rol es project manager. Estas respondiendo a cuestiones relativas a Seguros\
#     particularmente preguntas relacionadas con las funcionalidades de seguros y técnicas del producto 'tronweb'. \
#     tu responsabilidad es facilitar la resolución de incidencias valorando en las soluciones aportadas, una solucion simple, practica y rápida. \
#     Se profesional, comprometido y lider, estas tratando con compañeros de trabajo que, no tienen experiencia en Seguros ni tienen experiencia técnica \
#     con el producto asegurador tronweb \
#     Si no sabes la respuesta, dilo abiertamente"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"

# sustituir con la ruta donde dejaré el summary de tronweb y la carpeta con toda la documentación
#from pathlib import Path
# system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"

evaluator_system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [10]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [10]:
# def evaluator_user_prompt(reply, message, history):
#     user_prompt = f"Aqui esta la conversación entre el usuario y el agente: \n\n{history}\n\n"
#     user_prompt += f"Aqui está el último mensaje del usuario: \n\n{message}\n\n"
#     user_prompt += f"Aque está la última respuesta del agente: \n\n{reply}\n\n"
#     user_prompt += "Evalua la respuesta, contestando si es correcta o no y tu opinión"
#     return user_prompt


In [18]:
import os
from openai import api_key

gemini = OpenAI(
    api_key = os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

In [ ]:
#evaluo la contestacion, el mensaje y el historial de la conversacion, devolviendo un objeto Evaluation

# class Evaluation(BaseModel):
#     is_acceptable: bool
#     feedback: str
def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + \
               [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed


In [ ]:

#formo la entrada
messages = [{"role": "system", "content": evaluator_system_prompt}] + \
    [{"role": "user", "content": "did you work in equifax"}]
#lanzo la llamada al modelo de chatgpt 
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
#guardo la contestación
reply = response.choices[0].message.content

In [ ]:
evaluate(reply, "do you hold a patent?", messages[:1])
#[:1] → significa “desde el inicio hasta el índice 1 (sin incluirlo)”

In [ ]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()